# アフィ縦動画のジャンル別テンプレート

美容スキンケア / ドッグフード / 見守りカメラ / 婚活。伸びてる群の型を1枚にする。投稿も動画生成もしない。

`DATE` を変えれば、そのフォルダの `accounts.csv` で計算し直す。ローカルにファイルがあればそれを使い、無ければこのブランチの raw URL から取る。


In [ ]:
#@title 設定
BRANCH = "cursor/affi-genre-templates-6dc5"  #@param {type:"string"}
DATE = "2026-10-03"  #@param {type:"string"}
STRONG_MIN = 10  #@param {type:"integer"}
COMPARE_MIN = 3  #@param {type:"integer"}
THIN_BELOW = 10  #@param {type:"integer"}
MIN_GAP_PT = 10  #@param {type:"integer"}

from pathlib import Path
import sys
import urllib.request

REPO = "fireworker011/Research"
LOCAL_ROOT = Path("research/affi-templates")
if not LOCAL_ROOT.is_dir():
    here = Path.cwd()
    for candidate in [here, *here.parents]:
        if (candidate / "research/affi-templates/affi_genre_templates.py").is_file():
            LOCAL_ROOT = candidate / "research/affi-templates"
            break
RAW = f"https://raw.githubusercontent.com/{REPO}/{BRANCH}/research/affi-templates"
DOWNLOAD = Path("/content/affi-templates") if Path("/content").is_dir() else Path(".affi-templates-download")

def _fetch(rel: str, dest: Path) -> None:
    dest.parent.mkdir(parents=True, exist_ok=True)
    url = f"{RAW}/{rel}"
    print("get", url)
    urllib.request.urlretrieve(url, dest)

local_py = LOCAL_ROOT / "affi_genre_templates.py"
local_csv = LOCAL_ROOT / "data" / DATE / "accounts.csv"
if local_py.is_file():
    sys.path.insert(0, str(LOCAL_ROOT))
    print("module", local_py)
else:
    _fetch("affi_genre_templates.py", DOWNLOAD / "affi_genre_templates.py")
    sys.path.insert(0, str(DOWNLOAD))
    print("module", DOWNLOAD / "affi_genre_templates.py")

if local_csv.is_file():
    DATA_DIR = local_csv.parent
    print("data", DATA_DIR)
else:
    DATA_DIR = DOWNLOAD / "data" / DATE
    _fetch(f"data/{DATE}/accounts.csv", DATA_DIR / "accounts.csv")
    try:
        _fetch(f"data/{DATE}/videos.csv", DATA_DIR / "videos.csv")
    except Exception as exc:
        print("videos.csv は無し", exc)
    print("data", DATA_DIR)


In [ ]:
import affi_genre_templates as agt

try:
    from IPython.display import Markdown, display
except ImportError:
    class Markdown:
        def __init__(self, data):
            self.data = data
        def __repr__(self):
            return self.data
    def display(obj):
        print(getattr(obj, "data", obj))

built = agt.load_snapshot(
    DATA_DIR,
    agt.Thresholds(
        strong_min=STRONG_MIN,
        compare_min=COMPARE_MIN,
        thin_below=THIN_BELOW,
        min_gap_pt=MIN_GAP_PT,
    ),
)
out = LOCAL_ROOT / "templates" if local_csv.is_file() else DOWNLOAD / "templates"
written = agt.write_outputs(built, out)
print("wrote", len(written), "files to", out)
display(Markdown(agt.render_compare(built)))
for item in built.genres:
    display(Markdown(agt.render_genre(item)))
